In [1]:
from pynq import Overlay, allocate
import pynq.lib.dma
import numpy as np

# load overlay
ol = Overlay('./design_1_AXISShift.bit')
ol?

In [2]:
dma = ol.axi_dma_0 # load DMA

In [3]:
N = 1024
in_buf = allocate(shape=(N,), dtype=np.uint32) # physically contiguous, DMA-safe
out_buf = allocate(shape=(N,), dtype=np.uint32)

# fill the input with arbitrary test data
for i in range(N):
    in_buf[i] = i * 7 + 3
out_buf[:] = 0
print(in_buf)
print(out_buf)

[   3   10   17 ..., 7150 7157 7164]
[0 0 0 ..., 0 0 0]


In [4]:
# run the loopback: PS -> DMA(MM2S)
dma.sendchannel.transfer(in_buf) # send channel feeds the IP input
dma.recvchannel.transfer(out_buf) # recieve channel captures the IP input
dma.sendchannel.wait()
dma.recvchannel.wait()

In [5]:
print(out_buf)

[   1    5    8 ..., 3575 3578 3582]


In [6]:
# a compact vectorized check using numpy
expected = in_buf >> 1
assert np.array_equal(out_buf, expected), "Mismatch detected"
print(f"All {N} words shifted correctly.")

All 1024 words shifted correctly.


In [7]:
# Release the contiguous buffers when finished
in_buf.freebuffer()
out_buf.freebuffer()